In [ ]:
#!/usr/bin/env python3
"""
Calculate residue-level C1' RMSF for multiple systems and replicas.

Each trajectory is aligned to the first frame of replica 1 using:
    name P or name C1' or name C4'

RMSF is calculated using one C1' atom per nucleotide.

Example:
    python calculate_rmsf.py \
        --manifest configs/systems/rmsf_k_parallel.local.csv \
        --output data/processed/rmsf/rmsf_k_parallel.npz
"""

import argparse
import csv
from collections import OrderedDict
from pathlib import Path

import mdtraj as md
import numpy as np


ALIGN_SELECTION = "name P or name C1' or name C4'"
RMSF_SELECTION = "name C1'"


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Calculate C1' RMSF across MD replicas."
    )

    parser.add_argument(
        "--manifest",
        required=True,
        type=Path,
        help="CSV file containing topology and trajectory paths.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output NPZ file.",
    )

    return parser.parse_args()


def resolve_path(path_text, manifest_directory):
    """將 manifest 中的相對路徑轉換成完整路徑。"""

    path = Path(path_text.strip())

    if path.is_absolute():
        return path

    return manifest_directory / path


def load_manifest(manifest_path):
    """讀取並依照 system_id 整理系統資料。"""

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"找不到 manifest：{manifest_path}"
        )

    required_columns = {
        "system_id",
        "label",
        "color",
        "topology",
        "replica",
        "trajectory",
    }

    systems = OrderedDict()
    manifest_directory = manifest_path.parent

    with manifest_path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        if reader.fieldnames is None:
            raise ValueError("Manifest 沒有標題列。")

        missing_columns = required_columns - set(reader.fieldnames)

        if missing_columns:
            raise ValueError(
                "Manifest 缺少欄位："
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            system_id = row["system_id"].strip()

            if not system_id:
                continue

            topology_path = resolve_path(
                row["topology"],
                manifest_directory,
            )

            trajectory_path = resolve_path(
                row["trajectory"],
                manifest_directory,
            )

            systems.setdefault(
                system_id,
                {
                    "label": row["label"].strip(),
                    "color": row["color"].strip(),
                    "topology": topology_path,
                    "replicas": [],
                },
            )

            # 同一個系統應該使用相同 topology
            if systems[system_id]["topology"] != topology_path:
                raise ValueError(
                    f"{system_id} 使用了不一致的 topology。"
                )

            systems[system_id]["replicas"].append(
                {
                    "replica": int(row["replica"]),
                    "trajectory": trajectory_path,
                }
            )

    return systems


def prepare_atom_selections(topology_path):
    """
    建立 alignment 與 RMSF 原子選擇。

    為了減少記憶體使用，只會從 DCD 載入實際需要的原子。
    """

    if not topology_path.exists():
        raise FileNotFoundError(
            f"找不到 topology：{topology_path}"
        )

    topology = md.load_topology(str(topology_path))

    align_full_indices = topology.select(
        ALIGN_SELECTION
    )

    rmsf_full_indices = topology.select(
        RMSF_SELECTION
    )

    if len(align_full_indices) == 0:
        raise ValueError(
            f"找不到 alignment atoms：{ALIGN_SELECTION}"
        )

    if len(rmsf_full_indices) == 0:
        raise ValueError(
            f"找不到 RMSF atoms：{RMSF_SELECTION}"
        )

    # 只載入 alignment 或 RMSF 計算需要的原子
    keep_full_indices = np.asarray(
        sorted(
            set(align_full_indices)
            | set(rmsf_full_indices)
        ),
        dtype=int,
    )

    # 建立原始 atom index 到切片後 index 的對照
    index_mapping = {
        original_index: sliced_index
        for sliced_index, original_index
        in enumerate(keep_full_indices)
    }

    align_sliced_indices = np.asarray(
        [
            index_mapping[index]
            for index in align_full_indices
        ],
        dtype=int,
    )

    rmsf_sliced_indices = np.asarray(
        [
            index_mapping[index]
            for index in rmsf_full_indices
        ],
        dtype=int,
    )

    residue_labels = np.asarray(
        [
            (
                f"{topology.atom(index).residue.name}"
                f"{topology.atom(index).residue.resSeq}"
            )
            for index in rmsf_full_indices
        ],
        dtype=str,
    )

    return {
        "keep_indices": keep_full_indices,
        "align_indices": align_sliced_indices,
        "rmsf_indices": rmsf_sliced_indices,
        "labels": residue_labels,
    }


def calculate_rmsf(
    trajectory_path,
    topology_path,
    reference_frame,
    selections,
):
    """
    計算單一 replica 的 C1' RMSF。

    軌跡會先使用核酸骨架原子對齊至 replica 1 的第一幀，
    再計算每個 C1' 原子相對於時間平均位置的 RMSF。
    """

    if not trajectory_path.exists():
        raise FileNotFoundError(
            f"找不到 trajectory：{trajectory_path}"
        )

    trajectory = md.load(
        str(trajectory_path),
        top=str(topology_path),
        atom_indices=selections["keep_indices"],
    )

    trajectory.superpose(
        reference_frame,
        frame=0,
        atom_indices=selections["align_indices"],
        ref_atom_indices=selections["align_indices"],
    )

    coordinates = trajectory.xyz[
        :,
        selections["rmsf_indices"],
        :,
    ]

    # 計算每個 C1' 原子的時間平均座標
    mean_coordinates = np.mean(
        coordinates,
        axis=0,
    )

    squared_displacement = np.sum(
        (coordinates - mean_coordinates) ** 2,
        axis=2,
    )

    # MDTraj 座標單位為 nm
    rmsf_nm = np.sqrt(
        np.mean(
            squared_displacement,
            axis=0,
        )
    )

    # 轉換為 Å
    rmsf_angstrom = rmsf_nm * 10.0

    return rmsf_angstrom, trajectory.n_frames


def process_system(system_id, system_info):
    """處理單一系統的所有 replicas。"""

    topology_path = system_info["topology"]

    selections = prepare_atom_selections(
        topology_path
    )

    replicas = sorted(
        system_info["replicas"],
        key=lambda item: item["replica"],
    )

    if not replicas:
        print(f"[跳過] {system_id} 沒有 replica。")
        return None

    # 使用 replica 1 的第一幀作為共同 alignment reference
    reference_trajectory = replicas[0]["trajectory"]

    if not reference_trajectory.exists():
        print(
            f"[跳過] Reference trajectory 不存在："
            f"{reference_trajectory}"
        )
        return None

    reference_frame = md.load_frame(
        str(reference_trajectory),
        index=0,
        top=str(topology_path),
        atom_indices=selections["keep_indices"],
    )

    replica_rmsf = []
    valid_replica_ids = []

    print(f"\n正在處理：{system_info['label']}")

    for replica_info in replicas:
        replica_id = replica_info["replica"]
        trajectory_path = replica_info["trajectory"]

        try:
            rmsf_values, frame_count = calculate_rmsf(
                trajectory_path=trajectory_path,
                topology_path=topology_path,
                reference_frame=reference_frame,
                selections=selections,
            )

        except (OSError, ValueError) as error:
            print(
                f"  [警告] Replica {replica_id} 失敗：{error}"
            )
            continue

        replica_rmsf.append(rmsf_values)
        valid_replica_ids.append(replica_id)

        print(
            f"  Replica {replica_id}: "
            f"{frame_count} frames, "
            f"mean RMSF={np.mean(rmsf_values):.3f} Å"
        )

    if not replica_rmsf:
        print(f"  [跳過] {system_id} 沒有有效結果。")
        return None

    return {
        "rmsf": np.vstack(replica_rmsf),
        "replica_ids": np.asarray(
            valid_replica_ids,
            dtype=int,
        ),
        "labels": selections["labels"],
    }


def main():
    """主程式。"""

    args = parse_arguments()

    systems = load_manifest(
        args.manifest
    )

    results = {}
    replica_ids = {}
    system_meta = {}
    common_labels = None

    print("Starting C1' RMSF calculation")

    for system_id, system_info in systems.items():
        system_result = process_system(
            system_id,
            system_info,
        )

        if system_result is None:
            continue

        current_labels = system_result["labels"]

        if common_labels is None:
            common_labels = current_labels

        elif not np.array_equal(
            common_labels,
            current_labels,
        ):
            raise ValueError(
                f"{system_id} 的 residue labels 與其他系統不一致。"
            )

        # shape: replicas × residues，尚未跨 replicas 平均
        results[system_id] = system_result["rmsf"]
        replica_ids[system_id] = system_result["replica_ids"]

        system_meta[system_id] = {
            "label": system_info["label"],
            "color": system_info["color"],
        }

    if not results:
        raise RuntimeError(
            "沒有任何 RMSF 結果可以儲存。"
        )

    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        args.output,
        results=results,
        labels=common_labels,
        system_meta=system_meta,
        replica_ids=replica_ids,
        unit=np.asarray("angstrom"),
        atom_selection=np.asarray(RMSF_SELECTION),
        alignment_selection=np.asarray(ALIGN_SELECTION),
    )

    print(f"\nRMSF data saved to: {args.output}")


if __name__ == "__main__":
    main()